# LLM Architecture Study: GPT-2 vs BERT

This notebook covers the two foundational architectures required by the project:
- **GPT-2** — Decoder-only transformer (autoregressive generation)
- **BERT** — Encoder-only transformer (bidirectional understanding)

We load both from HuggingFace, inspect their internals, and connect them to their roles in the RAG pipeline.

In [ ]:
# Install if needed
# !pip install transformers torch sentence-transformers

## Part 1 — GPT-2: Decoder-only Transformer

### Key properties
- **Architecture**: Stacked transformer **decoder** blocks
- **Attention type**: Causal (unidirectional) — each token attends only to previous tokens
- **Pre-training task**: Autoregressive language modeling — predict the next token P(t_n | t_1 … t_{n-1})
- **Strength**: Natural fit for text generation; scales to GPT-3, GPT-4, Claude, Llama
- **Weakness**: Cannot see future context — limits understanding tasks

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Load GPT-2 (smallest version: 117M params)
tokenizer_gpt2 = GPT2Tokenizer.from_pretrained('gpt2')
model_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')
model_gpt2.eval()

print(f"GPT-2 parameters: {sum(p.numel() for p in model_gpt2.parameters()):,}")
print(f"Number of transformer layers: {model_gpt2.config.n_layer}")
print(f"Hidden size: {model_gpt2.config.n_embd}")
print(f"Attention heads: {model_gpt2.config.n_head}")
print(f"Vocabulary size: {model_gpt2.config.vocab_size}")

In [ ]:
# Inspect the causal attention mask
# GPT-2 uses a lower-triangular mask so each position only attends to prior positions
seq_len = 6
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("Causal (lower-triangular) attention mask for seq_len=6:")
print(causal_mask.int().numpy())
print("\n1 = can attend, 0 = masked out (cannot look ahead)")

In [ ]:
# Generate text with GPT-2
prompt = "Retrieval-Augmented Generation is a technique that"
inputs = tokenizer_gpt2.encode(prompt, return_tensors='pt')

with torch.no_grad():
    output = model_gpt2.generate(
        inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer_gpt2.eos_token_id,
    )

generated = tokenizer_gpt2.decode(output[0], skip_special_tokens=True)
print("Generated text:")
print(generated)

In [ ]:
# Inspect per-token logits to understand autoregressive prediction
test_input = tokenizer_gpt2.encode("The capital of France is", return_tensors='pt')

with torch.no_grad():
    outputs = model_gpt2(test_input)
    logits = outputs.logits  # shape: (batch, seq_len, vocab_size)

# The last position predicts the next token
next_token_logits = logits[0, -1, :]  # (vocab_size,)
top5_ids = next_token_logits.topk(5).indices
top5_tokens = [tokenizer_gpt2.decode([i]) for i in top5_ids]
top5_probs = torch.softmax(next_token_logits, dim=-1)[top5_ids]

print("Top-5 next-token predictions for 'The capital of France is':")
for token, prob in zip(top5_tokens, top5_probs):
    print(f"  {repr(token):20s}  p={prob:.4f}")

## Part 2 — BERT: Encoder-only Transformer

### Key properties
- **Architecture**: Stacked transformer **encoder** blocks
- **Attention type**: Bidirectional (full self-attention) — every token attends to every other token
- **Pre-training tasks**: Masked Language Modeling (MLM) + Next Sentence Prediction (NSP)
- **Strength**: Rich contextual embeddings — the same word gets different representations in different contexts
- **Weakness**: Cannot generate text autoregressively

### Connection to RAG
The `sentence-transformers` library fine-tunes BERT variants for **semantic similarity** — making them ideal for embedding documents and queries in a shared vector space.

In [ ]:
from transformers import BertTokenizer, BertModel

tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')
model_bert = BertModel.from_pretrained('bert-base-uncased')
model_bert.eval()

print(f"BERT parameters: {sum(p.numel() for p in model_bert.parameters()):,}")
print(f"Number of transformer layers: {model_bert.config.num_hidden_layers}")
print(f"Hidden size: {model_bert.config.hidden_size}")
print(f"Attention heads: {model_bert.config.num_attention_heads}")

In [ ]:
# Demonstrate bidirectional attention — BERT sees ALL positions
seq_len = 6
full_attention = torch.ones(seq_len, seq_len)
print("Full (bidirectional) attention mask for seq_len=6:")
print(full_attention.int().numpy())
print("\nEvery position can attend to every other position.")

In [ ]:
# Show contextual embeddings: same word, different context → different vector
sentences = [
    "The bank approved the loan.",
    "We sat on the river bank.",
]

embeddings = []
for sent in sentences:
    enc = tokenizer_bert(sent, return_tensors='pt')
    with torch.no_grad():
        out = model_bert(**enc)
    # Find the token index for 'bank'
    tokens = tokenizer_bert.convert_ids_to_tokens(enc['input_ids'][0])
    bank_idx = tokens.index('bank')
    bank_emb = out.last_hidden_state[0, bank_idx]  # (hidden_size,)
    embeddings.append(bank_emb)
    print(f"Sentence: {sent}")
    print(f"  'bank' token index: {bank_idx}")
    print(f"  embedding norm: {bank_emb.norm():.3f}")

# Cosine similarity between the two 'bank' embeddings
cos_sim = torch.nn.functional.cosine_similarity(
    embeddings[0].unsqueeze(0),
    embeddings[1].unsqueeze(0)
).item()
print(f"\nCosine similarity between 'bank' in sentence 1 vs 2: {cos_sim:.4f}")
print("(< 1.0 confirms the embeddings differ based on context — bidirectionality at work)")

## Part 3 — Sentence-Transformers: BERT for Retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# all-MiniLM-L6-v2 = fine-tuned BERT variant optimized for semantic similarity
st_model = SentenceTransformer('all-MiniLM-L6-v2')

docs = [
    "RAG retrieves relevant passages and injects them into the LLM context.",
    "BERT is an encoder-only transformer pre-trained with masked language modeling.",
    "The Eiffel Tower is located in Paris, France.",
]
query = "How does retrieval-augmented generation work?"

doc_embeddings = st_model.encode(docs, normalize_embeddings=True)
query_embedding = st_model.encode(query, normalize_embeddings=True)

# Cosine similarities (dot product of normalized vectors)
sims = doc_embeddings @ query_embedding

print(f"Query: {query}\n")
for doc, sim in zip(docs, sims):
    print(f"  [{sim:.4f}] {doc}")
print(f"\nTop match: {docs[np.argmax(sims)]}")

## Part 4 — GPT-2 vs BERT: Comparison Table

| Property | GPT-2 (Decoder) | BERT (Encoder) |
|---|---|---|
| Architecture | Decoder-only | Encoder-only |
| Attention | Causal (unidirectional) | Bidirectional (full) |
| Pre-training task | Next token prediction | MLM + NSP |
| Output | Token probability distribution | Contextual token embeddings |
| Primary strength | Text generation | Text understanding / retrieval |
| Role in RAG | **Generator** (Component 5) | **Retriever** via sentence-transformers (Component 3) |
| Can generate fluently? | ✅ Yes | ❌ No |
| Context-sensitive embeddings? | Partial | ✅ Full |

### Why BERT for retrieval, GPT for generation?

**BERT's bidirectionality** means every token embedding is influenced by its full context on both sides. This produces the richest possible semantic representations — ideal for encoding documents and queries into a shared vector space where similarity search is meaningful.

**GPT-2's causal masking** makes it a natural token-by-token text generator — each output token is conditioned on all previous tokens. This is the correct inductive bias for generation: you produce text left to right.

In the RAG pipeline:
- **Embedding step** (Component 3) uses a **BERT variant** (sentence-transformers) to encode chunks and queries into 384-d vectors.
- **Generation step** (Component 5) uses a **GPT-style model** (Claude / GPT-4 / Llama) to produce fluent, grounded answers from the retrieved context.